# 08 · Weighted Fusion

When you deliberately want one retriever to matter more — after normalizing scores.

**Analogy handbook:** [weighted-fusion](../retriever-analogy-handbook.html#weighted-fusion)  
**Prerequisite:** run `00_basics_concepts.ipynb` once (or the setup cells below) so the Chroma index exists.

### Learning loop
1. Skim the analogy for this technique  
2. Run setup (reuse index if possible)  
3. Run the practical cells  
4. Ask: *Did this fix the failure mode, or only reshuffle noise?*


## Shared setup

These cells install packages, load the Llama 2 PDF, build/load the Chroma index, and define helpers.

> Prefer `REBUILD_INDEX = False` after the first successful build so later method notebooks reuse the same store.


### Learning: !pip install langchain_community langchain_text_splitters langchain_op

**What you'll learn:** Install the packages this notebook needs.

**What this cell does:** Installs required Python packages into the runtime.

**Watch for:** Run once; restart runtime if Colab asks.



In [ ]:
!pip install langchain_community langchain_text_splitters langchain_openai langchain_chroma pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 378.1/378.1 kB 2.2 MB/s eta 0:00:00


### Learning: IMPORTS

**What you'll learn:** Bring in LangChain, embeddings, and vector-store modules.

**What this cell does:** Runs `IMPORTS` and prints intermediate results you can inspect.

**Watch for:** If an import fails, re-run the install cell.



In [ ]:
print("All imports and setup starting...")

# ============================================================
# 1. IMPORTS
# ============================================================

from pathlib import Path
import getpass
import os
import shutil

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma

from langchain_classic.chains.hyde.base import (
    HypotheticalDocumentEmbedder
)

All imports and setup starting...


/tmp/ipykernel_520/2286258816.py:12: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


### Learning: from google.colab import userdata

**What you'll learn:** Bring in LangChain, embeddings, and vector-store modules.

**What this cell does:** Runs `from google.colab import userdata` and prints intermediate results you can inspect.

**Watch for:** If an import fails, re-run the install cell.



In [ ]:
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

### Learning: OPENAI API KEY

**What you'll learn:** Authenticate so embedding and chat calls can run.

**What this cell does:** Runs `OPENAI API KEY` and prints intermediate results you can inspect.

**Watch for:** Never hardcode secrets in shared notebooks.



In [ ]:
# ============================================================
# 2. OPENAI API KEY
# ============================================================

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass(
        "Enter your OpenAI API key: "
    )

print("OpenAI API key configured successfully.")

OpenAI API key configured successfully.


### Learning: DATA DIRECTORY

**What you'll learn:** Locate and load the Llama 2 paper as Document pages.

**What this cell does:** Runs `DATA DIRECTORY` and prints intermediate results you can inspect.

**Watch for:** Confirm page count and first-page text look sane.



In [ ]:
# ============================================================
# 3. DATA DIRECTORY
# ============================================================

DATA_DIR = Path(
    r"/content/"
)

preferred_pdf = DATA_DIR / "llama2-research-paper.pdf"

### Learning: FIND PDF

**What you'll learn:** Locate and load the Llama 2 paper as Document pages.

**What this cell does:** Runs `FIND PDF` and prints intermediate results you can inspect.

**Watch for:** Confirm page count and first-page text look sane.



In [ ]:
# ============================================================
# 4. FIND PDF
# ============================================================

if preferred_pdf.exists():

    PDF_PATH = preferred_pdf

else:

    available_pdfs = sorted(
        DATA_DIR.glob("*.pdf")
    )

    if len(available_pdfs) == 1:

        PDF_PATH = available_pdfs[0]

    elif len(available_pdfs) == 0:

        raise FileNotFoundError(
            f"No PDF file was found inside:\n{DATA_DIR}"
        )

    else:

        raise RuntimeError(
            "Multiple PDF files were found. "
            "Please set PDF_PATH manually.\n"
            + "\n".join(
                str(path)
                for path in available_pdfs
            )
        )


print("PDF found:")
print(PDF_PATH)

PDF found:
/content/llama2-research-paper.pdf


### Learning: LOAD PDF

**What you'll learn:** Locate and load the Llama 2 paper as Document pages.

**What this cell does:** Runs `LOAD PDF` and prints intermediate results you can inspect.

**Watch for:** Confirm page count and first-page text look sane.



In [15]:
# ============================================================
# 5. LOAD PDF
# ============================================================

loader = PyPDFLoader(
    str(PDF_PATH)
)

pages = loader.load()

print(
    f"\nTotal PDF pages loaded: {len(pages)}"
)


# ============================================================
# 6. INSPECT FIRST PAGE
# ============================================================

print("\nFirst-page metadata:")
print(
    pages[0].metadata
)

print("\nFirst 1,000 characters:")
print(
    pages[0].page_content[:1000]
)


Total PDF pages loaded: 77

First-page metadata:
{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '/content/llama2-research-paper.pdf', 'total_pages': 77, 'page': 0, 'page_label': '1'}

First 1,000 characters:
Llama 2: Open Foundation and Fine-Tuned Chat Models
Hugo Touvron∗ Louis Martin† Kevin Stone†
Peter Albert Amjad Almahairi Yasmine Babaei Nikolay Bashlykov Soumya Batra
Prajjwal Bhargava Shruti Bhosale Dan Bikel Lukas Blecher Cristian Canton Ferrer Moya Chen
Guillem Cucurull David Esiobu Jude Fernandes Jeremy Fu Wenyin Fu Brian Fuller
Cynthia Gao Vedanuj Goswami Naman Goyal Anthony Hartshorn Saghar Hosseini Rui Hou
Hakan Inan Marcin Kardas Viktor Kerkez Madian Khabsa Isabel Kloumann Art

### Learning: IDENTIFY PAPER SECTIONS

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** Defines helper logic for: IDENTIFY PAPER SECTIONS.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [ ]:
# ============================================================
# 7. IDENTIFY PAPER SECTIONS
# ============================================================

def identify_section(
    paper_page: int
) -> str:

    if 1 <= paper_page <= 2:
        return "front_matter"

    if 3 <= paper_page <= 4:
        return "introduction"

    if 5 <= paper_page <= 7:
        return "pretraining"

    if 8 <= paper_page <= 19:
        return "fine_tuning"

    if 20 <= paper_page <= 31:
        return "safety"

    if 32 <= paper_page <= 35:
        return "discussion"

    if paper_page == 36:
        return "conclusion"

    if 37 <= paper_page <= 45:
        return "references"

    if 46 <= paper_page <= 77:
        return "appendix"

    return "unknown"


# ============================================================
# 8. ADD METADATA
# ============================================================

for page_document in pages:

    page_index = int(
        page_document.metadata.get(
            "page",
            0
        )
    )

    paper_page = (
        page_index + 1
    )

    page_document.metadata.update(
        {
            "paper": "Llama 2",
            "organization": "Meta",
            "year": 2023,
            "document_type": "research_paper",
            "paper_page": paper_page,
            "section": identify_section(
                paper_page
            ),
            "access_level": "public",
        }
    )


print("\nMetadata after enrichment:")

for page_document in pages[:5]:

    print(
        page_document.metadata
    )



Metadata after enrichment:
{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'D:\\complete_content_new\\Full-Stack-GenAI-Bootcamp-1.0\\Class-37-08-Aug-2026-prompting\\data\\llama2-research-paper.pdf', 'total_pages': 77, 'page': 0, 'page_label': '1', 'paper': 'Llama 2', 'organization': 'Meta', 'year': 2023, 'document_type': 'research_paper', 'paper_page': 1, 'section': 'front_matter', 'access_level': 'public'}
{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5',

### Learning: TEXT SPLITTING

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** Runs `TEXT SPLITTING` and prints intermediate results you can inspect.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [ ]:
# ============================================================
# 9. TEXT SPLITTING
# ============================================================

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True,
)

chunks = text_splitter.split_documents(
    pages
)

print(
    f"\nTotal pages: {len(pages)}"
)

print(
    f"Total chunks: {len(chunks)}"
)


# ============================================================
# 10. ADD CHUNK IDs
# ============================================================

for chunk_number, chunk in enumerate(
    chunks
):

    paper_page = chunk.metadata.get(
        "paper_page",
        "unknown"
    )

    chunk.metadata[
        "chunk_id"
    ] = (
        f"llama2-page-"
        f"{paper_page}-"
        f"chunk-{chunk_number}"
    )


print("\nFirst chunk content:")

print(
    chunks[0].page_content[:1000]
)

print("\nFirst chunk metadata:")

print(
    chunks[0].metadata
)


Total pages: 77
Total chunks: 343

First chunk content:
Llama 2: Open Foundation and Fine-Tuned Chat Models
Hugo Touvron∗ Louis Martin† Kevin Stone†
Peter Albert Amjad Almahairi Yasmine Babaei Nikolay Bashlykov Soumya Batra
Prajjwal Bhargava Shruti Bhosale Dan Bikel Lukas Blecher Cristian Canton Ferrer Moya Chen
Guillem Cucurull David Esiobu Jude Fernandes Jeremy Fu Wenyin Fu Brian Fuller
Cynthia Gao Vedanuj Goswami Naman Goyal Anthony Hartshorn Saghar Hosseini Rui Hou
Hakan Inan Marcin Kardas Viktor Kerkez Madian Khabsa Isabel Kloumann Artem Korenev
Punit Singh Koura Marie-Anne Lachaux Thibaut Lavril Jenya Lee Diana Liskovich
Yinghai Lu Yuning Mao Xavier Martinet Todor Mihaylov Pushkar Mishra
Igor Molybog Yixin Nie Andrew Poulton Jeremy Reizenstein Rashi Rungta Kalyan Saladi
Alan Schelten Ruan Silva Eric Michael Smith Ranjan Subramanian Xiaoqing Ellen Tan Binh Tang
Ross Taylor Adina Williams Jian Xiang Kuan Puxin Xu Zheng Yan Iliyan Zarov Yuchen Zhang
Angela Fan Melanie Kambadur Shar

### Learning: CREATE EMBEDDING MODEL

**What you'll learn:** Authenticate so embedding and chat calls can run.

**What this cell does:** Runs `CREATE EMBEDDING MODEL` and prints intermediate results you can inspect.

**Watch for:** Never hardcode secrets in shared notebooks.



In [ ]:
# ============================================================
# 11. CREATE EMBEDDING MODEL
# ============================================================

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)


# ============================================================
# 12. TEST EMBEDDING MODEL
# ============================================================

test_vector = embeddings.embed_query(
    "What is Llama 2?"
)

print(
    f"\nEmbedding dimensions: "
    f"{len(test_vector)}"
)

print(
    f"First 10 values: "
    f"{test_vector[:10]}"
)



Embedding dimensions: 1536
First 10 values: [0.0027942657470703125, -0.0521240234375, -0.021087646484375, -0.055419921875, -0.026397705078125, 0.028961181640625, -0.002071380615234375, 0.034759521484375, -0.0164794921875, -0.0245208740234375]


### Learning: CHROMA CONFIGURATION

**What you'll learn:** Build or reload the vector index used by retrievers.

**What this cell does:** Runs `CHROMA CONFIGURATION` and prints intermediate results you can inspect.

**Watch for:** Use REBUILD_INDEX=False after the first successful build.



In [ ]:
# ============================================================
# 13. CHROMA CONFIGURATION
# ============================================================

PERSIST_DIRECTORY = (
    DATA_DIR
    / "chroma_llama2_retriever"
)

COLLECTION_NAME = (
    "llama2_retriever_demo"
)


### Learning: CREATE OR LOAD VECTOR STORE

**What you'll learn:** Build or reload the vector index used by retrievers.

**What this cell does:** Runs `CREATE OR LOAD VECTOR STORE` and prints intermediate results you can inspect.

**Watch for:** Use REBUILD_INDEX=False after the first successful build.



In [ ]:
# ============================================================
# 14. CREATE OR LOAD VECTOR STORE
# ============================================================

# True  = rebuild complete vector DB
# False = reuse existing vector DB

REBUILD_INDEX = True

### Learning: VERIFY VECTOR STORE

**What you'll learn:** Break pages into retrieval-sized chunks.

**What this cell does:** Runs `VERIFY VECTOR STORE` and prints intermediate results you can inspect.

**Watch for:** Chunk size trades precision vs context — inspect a sample.



In [ ]:
if REBUILD_INDEX:

    print(
        "\nRebuilding vector store..."
    )

    if PERSIST_DIRECTORY.exists():

        shutil.rmtree(
            PERSIST_DIRECTORY,
            ignore_errors=True
        )

    vector_store = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        collection_name=COLLECTION_NAME,
        persist_directory=str(
            PERSIST_DIRECTORY
        ),
        collection_configuration={
            "hnsw": {
                "space": "cosine"
            }
        },
    )

    print(
        "New vector store created."
    )

else:

    if not PERSIST_DIRECTORY.exists():

        print(
            "\nExisting vector DB "
            "not found."
        )

        print(
            "Creating a new vector store..."
        )

        vector_store = Chroma.from_documents(
            documents=chunks,
            embedding=embeddings,
            collection_name=COLLECTION_NAME,
            persist_directory=str(
                PERSIST_DIRECTORY
            ),
            collection_configuration={
                "hnsw": {
                    "space": "cosine"
                }
            },
        )

        print(
            "New vector store created."
        )

    else:

        print(
            "\nLoading existing "
            "vector store..."
        )

        vector_store = Chroma(
            collection_name=COLLECTION_NAME,
            embedding_function=embeddings,
            persist_directory=str(
                PERSIST_DIRECTORY
            ),
        )

        print(
            "Existing vector store "
            "loaded."
        )


# ============================================================
# 15. VERIFY VECTOR STORE
# ============================================================

stored_count = (
    vector_store
    ._collection
    .count()
)

print(
    f"\nStored chunks: "
    f"{stored_count}"
)

print(
    f"Persisted at: "
    f"{PERSIST_DIRECTORY}"
)



Rebuilding vector store...
New vector store created.

Stored chunks: 343
Persisted at: D:\complete_content_new\Full-Stack-GenAI-Bootcamp-1.0\Class-37-08-Aug-2026-prompting\data\chroma_llama2_retriever


Suppose query hai:

How was Llama 2-Chat aligned with human preferences
and what role did reward models play?

Hop 1 may search:

How was Llama 2-Chat aligned with human preferences?

and retrieve RLHF-related chunks.

Then LLM sees those chunks and realizes:

I also need information about reward models.

So it generates Hop 2 query:

How were reward models trained and used in Llama 2-Chat?

Then:

Hop 2 Query
     ↓
Retriever
     ↓
Reward-model documents

Finally:

Hop 1 evidence
       +
Hop 2 evidence
       ↓
Final Answer

## Weighted fusion

### Learning: WEIGHTED RRF / HYBRID RETRIEVAL PRACTICAL

**What you'll learn:** Install the packages this notebook needs.

**What this cell does:** Runs `WEIGHTED RRF / HYBRID RETRIEVAL PRACTICAL` and prints intermediate results you can inspect.

**Watch for:** Run once; restart runtime if Colab asks.



In [ ]:
# ============================================================
# WEIGHTED RRF / HYBRID RETRIEVAL PRACTICAL
# LangChain EnsembleRetriever
# ============================================================

# ------------------------------------------------------------
# 1. INSTALL
# ------------------------------------------------------------

# %pip install -U langchain-classic langchain-community rank-bm25


# ============================================================
# 2. IMPORTS
# ============================================================

from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

### Learning: CREATE SPARSE / BM25 RETRIEVER

**What you'll learn:** Break pages into retrieval-sized chunks.

**What this cell does:** Runs `CREATE SPARSE / BM25 RETRIEVER` and prints intermediate results you can inspect.

**Watch for:** Chunk size trades precision vs context — inspect a sample.



In [ ]:
# ============================================================
# 3. CREATE SPARSE / BM25 RETRIEVER
# ============================================================

bm25_retriever = BM25Retriever.from_documents(
    chunks
)

bm25_retriever.k = 5

print("BM25 retriever created.")

### Learning: CREATE DENSE / VECTOR RETRIEVER

**What you'll learn:** Retrieve by semantic nearest-neighbors.

**What this cell does:** Runs `CREATE DENSE / VECTOR RETRIEVER` and prints intermediate results you can inspect.

**Watch for:** Dense can miss exact codes/IDs — compare with BM25 later.



In [ ]:
# ============================================================
# 4. CREATE DENSE / VECTOR RETRIEVER
# ============================================================

dense_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 5
    }
)

print("Dense retriever created.")


### Learning: CREATE WEIGHTED HYBRID RETRIEVER

**What you'll learn:** Match exact tokens with lexical sparse retrieval.

**What this cell does:** Runs `CREATE WEIGHTED HYBRID RETRIEVER` and prints intermediate results you can inspect.

**Watch for:** Best for IDs, acronyms, and rare proper nouns.



In [ ]:
# ============================================================
# 5. CREATE WEIGHTED HYBRID RETRIEVER
# ============================================================

# IMPORTANT:
#
# LangChain EnsembleRetriever performs
# Weighted Reciprocal Rank Fusion (Weighted RRF).
#
# Here:
# BM25 weight  = 0.4
# Dense weight = 0.6

weighted_hybrid_retriever = EnsembleRetriever(
    retrievers=[
        bm25_retriever,
        dense_retriever
    ],
    weights=[
        0.4,
        0.6
    ]
)

print(
    "Weighted hybrid retriever created."
)


### Learning: USER QUERY

**What you'll learn:** Execute the next step in the retrieval pipeline and observe the output.

**What this cell does:** Runs `USER QUERY` and prints intermediate results you can inspect.

**Watch for:** Relate this step to find → order → trim in the RAG pipeline.



In [ ]:
# ============================================================
# 6. USER QUERY
# ============================================================

query = (
    "How does Llama 2 improve safety?"
)

### Learning: BM25 RESULTS

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** Executes retrieval/generation for: BM25 RESULTS.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [ ]:
# ============================================================
# 7. BM25 RESULTS
# ============================================================

bm25_results = bm25_retriever.invoke(
    query
)

print(
    "\nBM25 RESULTS"
)

print(
    "=" * 100
)

for i, document in enumerate(
    bm25_results,
    start=1
):

    print(
        f"\nRank {i}"
    )

    print(
        "Page:",
        document.metadata.get(
            "paper_page"
        )
    )

    print(
        "Chunk ID:",
        document.metadata.get(
            "chunk_id"
        )
    )

    print(
        document.page_content[:500]
    )
print("\nUSER QUERY:")
print(query)

### Learning: DENSE RESULTS

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** Executes retrieval/generation for: DENSE RESULTS.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [ ]:
# ============================================================
# 8. DENSE RESULTS
# ============================================================

dense_results = dense_retriever.invoke(
    query
)

print(
    "\n\nDENSE VECTOR RESULTS"
)

print(
    "=" * 100
)

for i, document in enumerate(
    dense_results,
    start=1
):

    print(
        f"\nRank {i}"
    )

    print(
        "Page:",
        document.metadata.get(
            "paper_page"
        )
    )

    print(
        "Chunk ID:",
        document.metadata.get(
            "chunk_id"
        )
    )

    print(
        document.page_content[:500]
    )

### Learning: WEIGHTED RRF RESULTS

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** Executes retrieval/generation for: WEIGHTED RRF RESULTS.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [ ]:
# ============================================================
# 9. WEIGHTED RRF RESULTS
# ============================================================

hybrid_results = (
    weighted_hybrid_retriever.invoke(
        query
    )
)

print(
    "\n\nWEIGHTED RRF RESULTS"
)

print(
    "=" * 100
)

for i, document in enumerate(
    hybrid_results,
    start=1
):

    print(
        f"\nFINAL RANK {i}"
    )

    print(
        "Page:",
        document.metadata.get(
            "paper_page"
        )
    )

    print(
        "Section:",
        document.metadata.get(
            "section"
        )
    )

    print(
        "Chunk ID:",
        document.metadata.get(
            "chunk_id"
        )
    )

    print(
        "-" * 100
    )

    print(
        document.page_content[:700]
    )


### Learning: FINAL FLOW

**What you'll learn:** Match exact tokens with lexical sparse retrieval.

**What this cell does:** Runs `FINAL FLOW` and prints intermediate results you can inspect.

**Watch for:** Best for IDs, acronyms, and rare proper nouns.



In [ ]:
# ============================================================
# 10. FINAL FLOW
# ============================================================

"""
                     USER QUERY
                         |
             -------------------------
             |                       |
             v                       v
       BM25 Retriever          Dense Retriever
             |                       |
             v                       v
        Ranked List A           Ranked List B
             |                       |
             -----------+-----------
                        |
                        v
               Weighted RRF

            BM25 Weight  = 0.4
            Dense Weight = 0.6

                        |
                        v
                Final Ranking
"""


print(
    "\nWeighted RRF practical completed."
)

### Learning: 2. Try Different Weights

**What you'll learn:** Execute the next step in the retrieval pipeline and observe the output.

**What this cell does:** Runs `2. Try Different Weights` and prints intermediate results you can inspect.

**Watch for:** Relate this step to find → order → trim in the RAG pipeline.



In [ ]:
2. Try Different Weights

This part is useful in class:

### Learning: TEST DIFFERENT WEIGHT COMBINATIONS

**What you'll learn:** Match exact tokens with lexical sparse retrieval.

**What this cell does:** Executes retrieval/generation for: TEST DIFFERENT WEIGHT COMBINATIONS.

**Watch for:** Best for IDs, acronyms, and rare proper nouns.



In [ ]:
# ============================================================
# TEST DIFFERENT WEIGHT COMBINATIONS
# ============================================================

weight_configs = {
    "50_50": [0.5, 0.5],
    "70_BM25_30_Dense": [0.7, 0.3],
    "30_BM25_70_Dense": [0.3, 0.7],
}


for config_name, weights in weight_configs.items():

    retriever = EnsembleRetriever(
        retrievers=[
            bm25_retriever,
            dense_retriever
        ],
        weights=weights
    )

    results = retriever.invoke(
        query
    )

    print(
        "\n" + "=" * 100
    )

    print(
        f"CONFIGURATION: {config_name}"
    )

    print(
        f"BM25 weight: {weights[0]}"
    )

    print(
        f"Dense weight: {weights[1]}"
    )

    print(
        "=" * 100
    )

    for rank, document in enumerate(
        results[:5],
        start=1
    ):

        print(
            rank,
            "| Page:",
            document.metadata.get(
                "paper_page"
            ),
            "| Chunk:",
            document.metadata.get(
                "chunk_id"
            )
        )

### Learning: TRUE SCORE-BASED WEIGHTED FUSION

**What you'll learn:** Bring in LangChain, embeddings, and vector-store modules.

**What this cell does:** Runs `TRUE SCORE-BASED WEIGHTED FUSION` and prints intermediate results you can inspect.

**Watch for:** If an import fails, re-run the install cell.



In [ ]:
# ============================================================
# TRUE SCORE-BASED WEIGHTED FUSION
# ============================================================

import numpy as np


# ============================================================
# 1. QUERY
# ============================================================

query = (
    "How does Llama 2 improve safety?"
)


### Learning: DENSE RESULTS WITH SCORES

**What you'll learn:** Execute the next step in the retrieval pipeline and observe the output.

**What this cell does:** Executes retrieval/generation for: DENSE RESULTS WITH SCORES.

**Watch for:** Relate this step to find → order → trim in the RAG pipeline.



In [ ]:
# ============================================================
# 2. DENSE RESULTS WITH SCORES
# ============================================================

dense_scored_results = (
    vector_store
    .similarity_search_with_relevance_scores(
        query=query,
        k=10
    )
)


### Learning: BUILD BM25 RETRIEVER INTERNALLY

**What you'll learn:** Bring in LangChain, embeddings, and vector-store modules.

**What this cell does:** Runs `BUILD BM25 RETRIEVER INTERNALLY` and prints intermediate results you can inspect.

**Watch for:** If an import fails, re-run the install cell.



In [ ]:
# ============================================================
# 3. BUILD BM25 RETRIEVER INTERNALLY
# ============================================================

from rank_bm25 import BM25Okapi


texts = [
    chunk.page_content
    for chunk in chunks
]


tokenized_corpus = [
    text.lower().split()
    for text in texts
]


bm25_model = BM25Okapi(
    tokenized_corpus
)


tokenized_query = (
    query.lower().split()
)


bm25_scores = bm25_model.get_scores(
    tokenized_query
)


### Learning: NORMALIZATION FUNCTION

**What you'll learn:** Execute the next step in the retrieval pipeline and observe the output.

**What this cell does:** Defines helper logic for: NORMALIZATION FUNCTION.

**Watch for:** Relate this step to find → order → trim in the RAG pipeline.



In [ ]:
# ============================================================
# 4. NORMALIZATION FUNCTION
# ============================================================

def min_max_normalize(
    values
):

    values = np.asarray(
        values,
        dtype=float
    )

    minimum = values.min()
    maximum = values.max()

    if maximum == minimum:
        return np.zeros_like(
            values
        )

    return (
        values - minimum
    ) / (
        maximum - minimum
    )


### Learning: NORMALIZE BM25 SCORES

**What you'll learn:** Match exact tokens with lexical sparse retrieval.

**What this cell does:** Runs `NORMALIZE BM25 SCORES` and prints intermediate results you can inspect.

**Watch for:** Best for IDs, acronyms, and rare proper nouns.



In [ ]:
# ============================================================
# 5. NORMALIZE BM25 SCORES
# ============================================================

normalized_bm25_scores = (
    min_max_normalize(
        bm25_scores
    )
)


### Learning: CREATE SCORE MAP

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** Runs `CREATE SCORE MAP` and prints intermediate results you can inspect.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [ ]:
# ============================================================
# 6. CREATE SCORE MAP
# ============================================================

bm25_score_map = {}


for chunk, score in zip(
    chunks,
    normalized_bm25_scores
):

    chunk_id = chunk.metadata.get(
        "chunk_id"
    )

    bm25_score_map[
        chunk_id
    ] = float(
        score
    )


### Learning: COLLECT DENSE CANDIDATES

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** Runs `COLLECT DENSE CANDIDATES` and prints intermediate results you can inspect.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [ ]:
# ============================================================
# 7. COLLECT DENSE CANDIDATES
# ============================================================

candidate_rows = []


for document, dense_score in (
    dense_scored_results
):

    chunk_id = (
        document.metadata.get(
            "chunk_id"
        )
    )

    bm25_score = (
        bm25_score_map.get(
            chunk_id,
            0.0
        )
    )

    candidate_rows.append(
        {
            "document": document,
            "chunk_id": chunk_id,
            "bm25_score": bm25_score,
            "dense_score": float(
                dense_score
            )
        }
    )

### Learning: APPLY WEIGHTS

**What you'll learn:** Execute the next step in the retrieval pipeline and observe the output.

**What this cell does:** Runs `APPLY WEIGHTS` and prints intermediate results you can inspect.

**Watch for:** Relate this step to find → order → trim in the RAG pipeline.



In [ ]:
# ============================================================
# 8. APPLY WEIGHTS
# ============================================================

BM25_WEIGHT = 0.4
DENSE_WEIGHT = 0.6


for row in candidate_rows:

    row["final_score"] = (
        BM25_WEIGHT
        * row["bm25_score"]
        +
        DENSE_WEIGHT
        * row["dense_score"]
    )


### Learning: SORT BY FINAL SCORE

**What you'll learn:** Execute the next step in the retrieval pipeline and observe the output.

**What this cell does:** Runs `SORT BY FINAL SCORE` and prints intermediate results you can inspect.

**Watch for:** Relate this step to find → order → trim in the RAG pipeline.



In [ ]:
# ============================================================
# 9. SORT BY FINAL SCORE
# ============================================================

candidate_rows = sorted(
    candidate_rows,
    key=lambda x: x[
        "final_score"
    ],
    reverse=True
)


### Learning: DISPLAY RESULTS

**What you'll learn:** Break pages into retrieval-sized chunks.

**What this cell does:** Runs `DISPLAY RESULTS` and prints intermediate results you can inspect.

**Watch for:** Chunk size trades precision vs context — inspect a sample.



In [ ]:


# ============================================================
# 10. DISPLAY RESULTS
# ============================================================

print(
    "\nTRUE WEIGHTED SCORE FUSION"
)

print(
    "=" * 100
)


for rank, row in enumerate(
    candidate_rows,
    start=1
):

    document = row[
        "document"
    ]

    print(
        f"\nRank {rank}"
    )

    print(
        "Chunk ID:",
        row["chunk_id"]
    )

    print(
        f"BM25 normalized score: "
        f"{row['bm25_score']:.4f}"
    )

    print(
        f"Dense score: "
        f"{row['dense_score']:.4f}"
    )

    print(
        f"Final weighted score: "
        f"{row['final_score']:.4f}"
    )

    print(
        document.page_content[:500]
    )